# Manual tries: Claude vs. Copilot

Compares the two manual result files in `data/manual_tries/` (one per model) on:
- how many restaurant menus each model found (`hasFoundMenu`)
- how many menu links the models agree on, and where/how they differ

In [3]:
import json
import pandas as pd
from foodscraper.config import DATA_DIR

In [6]:
manual_tries_dir = DATA_DIR / "manual_tries"
files = sorted(manual_tries_dir.glob("*.json"))

dfs = []
for f in files:
    model = f.stem.split("-", 1)[1]
    records = json.loads(f.read_text())
    d = pd.DataFrame(records)
    d["model"] = model
    dfs.append(d)

df = pd.concat(dfs, ignore_index=True)
models = sorted(df["model"].unique())
print(f"models: {models}")
df.head()

models: ['claude', 'copilot']


,id,name,hasFoundMenu,menuLink,format,source,model
0,ChIJRzRI2vNTUkYRRndb0XtBWow,Berlin Döner,True,https://wolt.com/en/dnk/copenhagen/restaurant/...,HTML,WOLT,claude
1,ChIJq9PWX55TUkYRBGa7u0DJYvE,Ata Pizza,True,https://ata-pizza.dk/,HTML,OFFICIAL_WEBSITE,claude
2,ChIJn4ah93RTUkYRzi8qRSKEaAI,Sticks'n'Sushi - Restaurant Vesterbro,True,https://a.storyblok.com/f/286316/x/a7b0fc6a7f/...,PDF,OFFICIAL_WEBSITE,claude
3,ChIJr-oJ-HRTUkYRf4fbGF1hGR4,Skipper's Bodega,False,,,,claude
4,ChIJcR5brHVTUkYR80HGten90uY,Auténtica Twuería Mexicana,False,,,,claude


## How many menus did each model find?

In [7]:
found_summary = (
    df.groupby("model")["hasFoundMenu"]
    .agg(n_restaurants="size", n_found="sum")
    .assign(found_rate=lambda d: d["n_found"] / d["n_restaurants"])
)
found_summary

,n_restaurants,n_found,found_rate
model,,,
claude,31,19,0.612903
copilot,31,18,0.580645


## Where do the models agree / disagree on `menuLink`?

Each restaurant is classified as:
- `agree` — both models found the same link
- `both_not_found` — neither model found a menu
- `one_found_only` — one model found a link, the other didn't
- `different_link` — both found a menu, but at different URLs

In [8]:
def norm(url):
    if not isinstance(url, str) or not url.strip():
        return None
    return url.strip().rstrip("/").lower()


def classify(row):
    a, b = norm(row[models[0]]), norm(row[models[1]])
    if a is None and b is None:
        return "both_not_found"
    if a == b:
        return "agree"
    if a is None or b is None:
        return "one_found_only"
    return "different_link"


comparison = df.pivot_table(
    index=["id", "name"], columns="model", values="menuLink", aggfunc="first"
).reindex(columns=models)
comparison["status"] = comparison.apply(classify, axis=1)

comparison["status"].value_counts()

status
different_link    10
both_not_found     9
one_found_only     7
agree              5
Name: count, dtype: int64

In [9]:
# restaurants where both models found the same menu link
comparison[comparison["status"] == "agree"]

,model,claude,copilot,status
id,name,,,
ChIJJ9NHpQ5TUkYRAhSfTMxPyaw,Grillaz,https://grillazcph.dk/,https://grillazcph.dk/,agree
ChIJXf3_RXRTUkYRXEJVH16cfPQ,Papa Ramen,https://www.paparamen.dk/bestil,https://www.paparamen.dk/bestil,agree
ChIJk5HSlzNTUkYRVuHEAOh6I4s,La Neta Vesterbro,https://www.laneta.dk/menu,https://www.laneta.dk/menu,agree
ChIJq9PWX55TUkYRBGa7u0DJYvE,Ata Pizza,https://ata-pizza.dk/,https://ata-pizza.dk/,agree
ChIJyUPdsXVTUkYR7Jtb_vDeMAs,Isted Grill,https://istedgrill.dk/menu/,https://istedgrill.dk/menu/,agree


In [10]:
disagreements = (
    comparison[comparison["status"].isin(["different_link", "one_found_only"])]
    .sort_values("status")
)

disagreements

,model,claude,copilot,status
id,name,,,
ChIJ6wYguKFTUkYRPiLMpWs4jsc,Grimal,https://grimal.dk/madogdrikke/menu,https://grimal.dk/,different_link
ChIJ7yTJtY1TUkYRrq0xrr2Rzyw,Börger,https://www.burger-vesterbro.dk/menu,https://www.burger-vesterbro.dk/,different_link
ChIJRzRI2vNTUkYRRndb0XtBWow,Berlin Döner,https://wolt.com/en/dnk/copenhagen/restaurant/...,https://berlindonercph.dk/menu/,different_link
ChIJb68b_sNTUkYRE-H3ITU1Nxw,Bar la Una,https://www.baruna.dk/s/1JulyFoodUna.pdf,https://www.baruna.dk/,different_link
ChIJd4d7unVTUkYR_1ErNvKNvYA,La Foretta,https://la-foretta.dk/la-foretta/delivery,https://laforetta.bestilonline.dk/,different_link
ChIJgTOK1Z1TUkYR6zeCXpFaE04,Pasha Kebab,https://pashakebab.dk/menu/,https://pashakebab.dk/,different_link
ChIJgUgIXCJTUkYROUvU9Ogpqmk,GAO Dumpling Bar,https://wolt.com/en/dnk/copenhagen/restaurant/...,https://gaodumpling.com/,different_link
ChIJmb256HVTUkYRKoxC2VhD9KM,Madhuset,https://www.madhusetvesterbro.dk/,http://www.madhusetvesterbro.dk/,different_link
ChIJn4ah93RTUkYRzi8qRSKEaAI,Sticks'n'Sushi - Restaurant Vesterbro,https://a.storyblok.com/f/286316/x/a7b0fc6a7f/...,https://www.sticksnsushi.com/dk/da/dine-in-menu/,different_link
